# Diagnóstico de enfermedades en hojas de papa y café
### Notebook de entrenamiento — Papa (PlantVillage) y Café (RoCoLe)

!Nota: Ejecute esto en google colab.

Este notebook entrena y compara **3 experimentos** por cultivo:
1. CNN simple desde cero (baseline)
2. MobileNetV2 — transfer learning (feature extraction)
3. MobileNetV2 — transfer learning con fine-tuning

Al final se guardan los mejores modelos y el mapeo de clases, listos para usar en la app Flask.

**Antes de correr:** sube tus datasets ya limpios a Google Drive con esta estructura:
```
MiDrive/proyecto_ia/data/potato/{Potato___Early_blight, Potato___Late_blight, Potato___healthy}
MiDrive/proyecto_ia/data/coffee/{coffee___healthy, coffee___rust, coffee___red_spider_mite}
```

Enlaces de donde se obtuvo los datasets:

Hojas de Papa(PlantVillage (subset "color", solo carpetas de papa) — Kaggle):
https://www.kaggle.com/datasets/abdallahalidev/plantvillage-dataset

Carpetas:
Potato___Early_blight, Potato___Late_blight, Potato___healthy

Hojas de Cafe(RoCoLe (Robusta Coffee Leaf Images) — Kaggle / Mendeley Data):
https://www.kaggle.com/code/nirmalsankalana/plant-disease-classification-train-dataset

Carpetas:
coffee___healthy, coffee___red_spider_mite, coffee___rust



## 1. Configuración inicial y montaje de Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import json
import shutil
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from sklearn.utils.class_weight import compute_class_weight
import seaborn as sns

print("TensorFlow version:", tf.__version__)

In [ ]:
BASE_DIR = '/content/drive/MyDrive/proyecto_ia'
MODELS_DIR = f'{BASE_DIR}/models'
os.makedirs(MODELS_DIR, exist_ok=True)

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
SEED = 123

CULTIVOS = {
    'papa': f'{BASE_DIR}/data/potato',
    'cafe': f'{BASE_DIR}/data/coffee',
}

## 2. Verificación rápida de datos (EDA mínimo)

Antes de entrenar, confirma cuántas imágenes hay por clase en cada cultivo.
Esto también te sirve para tu sección de EDA del README.

In [ ]:
for cultivo, ruta in CULTIVOS.items():
    print(f"\n=== {cultivo.upper()} ===")
    for clase in sorted(os.listdir(ruta)):
        n = len(os.listdir(os.path.join(ruta, clase)))
        print(f"  {clase}: {n} imágenes")

In [ ]:
# Visualiza algunos ejemplos por clase (útil para el README/EDA)
def mostrar_ejemplos(ruta_cultivo, n_por_clase=3):
    clases = sorted(os.listdir(ruta_cultivo))
    fig, axes = plt.subplots(len(clases), n_por_clase, figsize=(3*n_por_clase, 3*len(clases)))
    for i, clase in enumerate(clases):
        ruta_clase = os.path.join(ruta_cultivo, clase)
        archivos = os.listdir(ruta_clase)[:n_por_clase]
        for j, archivo in enumerate(archivos):
            img = tf.keras.utils.load_img(os.path.join(ruta_clase, archivo), target_size=IMG_SIZE)
            ax = axes[i, j] if len(clases) > 1 else axes[j]
            ax.imshow(img)
            ax.set_title(clase, fontsize=9)
            ax.axis('off')
    plt.tight_layout()
    plt.show()

mostrar_ejemplos(CULTIVOS['papa'])
mostrar_ejemplos(CULTIVOS['cafe'])

## 3. Funciones reutilizables (carga de datos, augmentation, modelos)

In [ ]:
def cargar_datasets(data_dir):
    """Carga train/val desde carpetas, retorna también los nombres de clase."""
    train_ds = tf.keras.utils.image_dataset_from_directory(
        data_dir, validation_split=0.2, subset="training",
        seed=SEED, image_size=IMG_SIZE, batch_size=BATCH_SIZE
    )
    val_ds = tf.keras.utils.image_dataset_from_directory(
        data_dir, validation_split=0.2, subset="validation",
        seed=SEED, image_size=IMG_SIZE, batch_size=BATCH_SIZE
    )
    class_names = train_ds.class_names

    # Dividimos val_ds en validation real + test (mitad y mitad)
    val_batches = tf.data.experimental.cardinality(val_ds)
    test_ds = val_ds.take(val_batches // 2)
    val_ds = val_ds.skip(val_batches // 2)

    # Prefetch para rendimiento
    train_ds = train_ds.prefetch(tf.data.AUTOTUNE)
    val_ds = val_ds.prefetch(tf.data.AUTOTUNE)
    test_ds = test_ds.prefetch(tf.data.AUTOTUNE)

    return train_ds, val_ds, test_ds, class_names

In [ ]:
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.15),
    layers.RandomZoom(0.15),
    layers.RandomBrightness(0.1),
], name="data_augmentation")

In [ ]:
def construir_cnn_desde_cero(num_clases):
    """Experimento 1: CNN simple entrenada desde cero (baseline)."""
    inputs = tf.keras.Input(shape=(224, 224, 3))
    x = data_augmentation(inputs)
    x = layers.Rescaling(1./255)(x)

    x = layers.Conv2D(32, 3, activation='relu')(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Conv2D(64, 3, activation='relu')(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Conv2D(128, 3, activation='relu')(x)
    x = layers.MaxPooling2D()(x)

    x = layers.Flatten()(x)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.4)(x)
    outputs = layers.Dense(num_clases, activation='softmax')(x)

    model = tf.keras.Model(inputs, outputs, name="cnn_desde_cero")
    model.compile(optimizer='adam',
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])
    return model

In [ ]:
def construir_mobilenet(num_clases, fine_tune=False, capas_a_descongelar=20):
    """Experimentos 2 y 3: MobileNetV2 con transfer learning.
    fine_tune=False -> solo feature extraction (base congelada)
    fine_tune=True  -> descongela las últimas capas para ajuste fino
    """
    base_model = tf.keras.applications.MobileNetV2(
        input_shape=(224, 224, 3), include_top=False, weights='imagenet'
    )
    base_model.trainable = fine_tune
    if fine_tune:
        for capa in base_model.layers[:-capas_a_descongelar]:
            capa.trainable = False

    inputs = tf.keras.Input(shape=(224, 224, 3))
    x = data_augmentation(inputs)
    x = tf.keras.applications.mobilenet_v2.preprocess_input(x)
    x = base_model(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(num_clases, activation='softmax')(x)

    model = tf.keras.Model(inputs, outputs, name=f"mobilenet_{'finetune' if fine_tune else 'feat_extract'}")

    lr = 1e-5 if fine_tune else 1e-3
    model.compile(optimizer=tf.keras.optimizers.Adam(lr),
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])
    return model, base_model

In [ ]:
def calcular_class_weights(train_ds):
    """Calcula pesos de clase para compensar desbalance."""
    labels = np.concatenate([y for x, y in train_ds], axis=0)
    clases_unicas = np.unique(labels)
    pesos = compute_class_weight('balanced', classes=clases_unicas, y=labels)
    return dict(zip(clases_unicas, pesos))

In [ ]:
def entrenar_modelo(model, train_ds, val_ds, nombre_run, class_weight=None,
                     epochs=20, patience=4):
    checkpoint_path = f'{MODELS_DIR}/checkpoint_{nombre_run}.keras'
    callbacks = [
        tf.keras.callbacks.ModelCheckpoint(
            filepath=checkpoint_path, save_best_only=True, monitor='val_accuracy'
        ),
        tf.keras.callbacks.EarlyStopping(
            monitor='val_accuracy', patience=patience, restore_best_weights=True
        ),
    ]
    history = model.fit(
        train_ds, validation_data=val_ds, epochs=epochs,
        callbacks=callbacks, class_weight=class_weight
    )
    return history

In [ ]:
def evaluar_modelo(model, test_ds, class_names, nombre_run):
    y_true, y_pred = [], []
    for x, y in test_ds:
        preds = model.predict(x, verbose=0)
        y_pred.extend(np.argmax(preds, axis=1))
        y_true.extend(y.numpy())

    y_true, y_pred = np.array(y_true), np.array(y_pred)
    acc = np.mean(y_true == y_pred)
    f1 = f1_score(y_true, y_pred, average='macro')

    print(f"\n=== {nombre_run} ===")
    print(f"Accuracy: {acc:.4f} | F1-macro: {f1:.4f}")
    print(classification_report(y_true, y_pred, target_names=class_names))

    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names)
    plt.title(f'Matriz de confusión - {nombre_run}')
    plt.ylabel('Real')
    plt.xlabel('Predicho')
    plt.tight_layout()
    plt.show()

    return {'nombre': nombre_run, 'accuracy': acc, 'f1_macro': f1}

In [ ]:
def graficar_historia(history, nombre_run):
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
    axes[0].plot(history.history['accuracy'], label='train')
    axes[0].plot(history.history['val_accuracy'], label='val')
    axes[0].set_title(f'Accuracy - {nombre_run}')
    axes[0].legend()

    axes[1].plot(history.history['loss'], label='train')
    axes[1].plot(history.history['val_loss'], label='val')
    axes[1].set_title(f'Loss - {nombre_run}')
    axes[1].legend()
    plt.tight_layout()
    plt.show()

## 4. Entrenamiento — Papa 🥔

Corremos los 3 experimentos para el cultivo de papa.

In [ ]:
train_papa, val_papa, test_papa, clases_papa = cargar_datasets(CULTIVOS['papa'])
print("Clases papa:", clases_papa)

class_weight_papa = calcular_class_weights(train_papa)
print("Class weights:", class_weight_papa)

In [ ]:
# Experimento 1: CNN desde cero
cnn_papa = construir_cnn_desde_cero(len(clases_papa))
hist_cnn_papa = entrenar_modelo(cnn_papa, train_papa, val_papa,
                                 'papa_cnn_scratch', class_weight=class_weight_papa)
graficar_historia(hist_cnn_papa, 'Papa - CNN desde cero')

In [ ]:
# Experimento 2: MobileNetV2 - feature extraction
mnet_papa, base_papa = construir_mobilenet(len(clases_papa), fine_tune=False)
hist_mnet_papa = entrenar_modelo(mnet_papa, train_papa, val_papa,
                                  'papa_mobilenet_feat', class_weight=class_weight_papa)
graficar_historia(hist_mnet_papa, 'Papa - MobileNetV2 (feature extraction)')

In [ ]:
# Experimento 3: MobileNetV2 - fine-tuning
# Partimos del modelo del experimento 2 y descongelamos las últimas capas
base_papa.trainable = True
for capa in base_papa.layers[:-20]:
    capa.trainable = False

mnet_papa.compile(optimizer=tf.keras.optimizers.Adam(1e-5),
                   loss='sparse_categorical_crossentropy', metrics=['accuracy'])

hist_finetune_papa = entrenar_modelo(mnet_papa, train_papa, val_papa,
                                      'papa_mobilenet_finetune',
                                      class_weight=class_weight_papa, epochs=10)
graficar_historia(hist_finetune_papa, 'Papa - MobileNetV2 (fine-tuning)')

In [ ]:
# Evaluación comparativa de los 3 experimentos - Papa
resultados_papa = []
resultados_papa.append(evaluar_modelo(cnn_papa, test_papa, clases_papa, 'Papa - CNN desde cero'))
resultados_papa.append(evaluar_modelo(mnet_papa, test_papa, clases_papa, 'Papa - MobileNetV2 fine-tuned'))

import pandas as pd
df_papa = pd.DataFrame(resultados_papa)
print(df_papa)

## 5. Entrenamiento — Café ☕

Repetimos exactamente el mismo proceso para café.

In [ ]:
train_cafe, val_cafe, test_cafe, clases_cafe = cargar_datasets(CULTIVOS['cafe'])
print("Clases café:", clases_cafe)

class_weight_cafe = calcular_class_weights(train_cafe)
print("Class weights:", class_weight_cafe)

In [ ]:
# Experimento 1: CNN desde cero
cnn_cafe = construir_cnn_desde_cero(len(clases_cafe))
hist_cnn_cafe = entrenar_modelo(cnn_cafe, train_cafe, val_cafe,
                                 'cafe_cnn_scratch', class_weight=class_weight_cafe)
graficar_historia(hist_cnn_cafe, 'Café - CNN desde cero')

In [ ]:
# Experimento 2: MobileNetV2 - feature extraction
mnet_cafe, base_cafe = construir_mobilenet(len(clases_cafe), fine_tune=False)
hist_mnet_cafe = entrenar_modelo(mnet_cafe, train_cafe, val_cafe,
                                  'cafe_mobilenet_feat', class_weight=class_weight_cafe)
graficar_historia(hist_mnet_cafe, 'Café - MobileNetV2 (feature extraction)')

In [ ]:
# Experimento 3: MobileNetV2 - fine-tuning
base_cafe.trainable = True
for capa in base_cafe.layers[:-20]:
    capa.trainable = False

mnet_cafe.compile(optimizer=tf.keras.optimizers.Adam(1e-5),
                   loss='sparse_categorical_crossentropy', metrics=['accuracy'])

hist_finetune_cafe = entrenar_modelo(mnet_cafe, train_cafe, val_cafe,
                                      'cafe_mobilenet_finetune',
                                      class_weight=class_weight_cafe, epochs=10)
graficar_historia(hist_finetune_cafe, 'Café - MobileNetV2 (fine-tuning)')

In [ ]:
# Evaluación comparativa de los 3 experimentos - Café
resultados_cafe = []
resultados_cafe.append(evaluar_modelo(cnn_cafe, test_cafe, clases_cafe, 'Café - CNN desde cero'))
resultados_cafe.append(evaluar_modelo(mnet_cafe, test_cafe, clases_cafe, 'Café - MobileNetV2 fine-tuned'))

df_cafe = pd.DataFrame(resultados_cafe)
print(df_cafe)

## 6. Guardar modelos finales y mapeo de clases

Guarda el mejor modelo por cultivo (normalmente el de fine-tuning) junto con
el orden de clases — ambos archivos los necesitarás en Flask.

In [ ]:
# Guardar modelo final de PAPA (ajusta si otro experimento dio mejor resultado)
mnet_papa.save(f'{MODELS_DIR}/modelo_papa.keras')
with open(f'{MODELS_DIR}/clases_papa.json', 'w') as f:
    json.dump(clases_papa, f)

# Guardar modelo final de CAFÉ
mnet_cafe.save(f'{MODELS_DIR}/modelo_cafe.keras')
with open(f'{MODELS_DIR}/clases_cafe.json', 'w') as f:
    json.dump(clases_cafe, f)

print("Modelos y mapeos de clases guardados en:", MODELS_DIR)

## 7. Tabla comparativa final (para el README)

Combina los resultados de ambos cultivos en una sola tabla — esto es
justo lo que pide la rúbrica: "al menos 3 experimentos con
tablas/gráficos comparativos".

In [ ]:
df_papa['cultivo'] = 'papa'
df_cafe['cultivo'] = 'cafe'
df_final = pd.concat([df_papa, df_cafe], ignore_index=True)
df_final = df_final[['cultivo', 'nombre', 'accuracy', 'f1_macro']]
print(df_final.to_string(index=False))

df_final.to_csv(f'{MODELS_DIR}/resultados_experimentos.csv', index=False)

## 8. Descargar los archivos necesarios para el proyecto Flask

Ejecuta esta celda al final para descargar los 4 archivos que necesitas
colocar en `backend/models/` de tu repositorio.

In [ ]:
from google.colab import files

for archivo in ['modelo_papa.keras', 'clases_papa.json',
                'modelo_cafe.keras', 'clases_cafe.json']:
    files.download(f'{MODELS_DIR}/{archivo}')

---
### Notas importantes

- **Preprocesamiento en Flask:** recuerda usar `tf.keras.applications.mobilenet_v2.preprocess_input`
  al procesar la imagen antes de predecir — el modelo fue entrenado esperando ese preprocesamiento,
  no una simple división entre 255.
- **Versión de TensorFlow:** anota la versión usada aquí (`tf.__version__`) y usa la misma
  (o una compatible) en tu `requirements.txt` de Flask.
- **Si el fine-tuning empeora los resultados** respecto a feature extraction (puede pasar con
  datasets pequeños como el de café), usa el modelo de feature extraction como final —
  documenta esa comparación en tu README, es un hallazgo válido y esperado por la rúbrica.
